In [1]:
import torch
import os
import platform

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(
        f"GPU memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )
else:
    print("WARNING: No GPU detected!")

print("=" * 60)

ENVIRONMENT CHECK
Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0
GPU memory: 14.56 GB


In [2]:
!pip install -q \
    transformers==4.57.1 \
    datasets==4.2.0 \
    peft==0.17.1 \
    accelerate==1.10.1 \
    bitsandbytes==0.48.1 \
    sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 113.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

In [3]:
import transformers
import datasets
import peft
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

Transformers: 4.57.1
Datasets: 4.2.0
PEFT: 0.17.1
Accelerate: 1.10.1
BitsAndBytes: 0.48.1


In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
import os

# Search your Google Drive for the files
targets = [
    "amharic_distillation_training_clean.csv",
    "eval_prompts_60.csv",
    "predictions_template.csv"
]

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file in targets:
            print(os.path.join(root, file))


/content/drive/MyDrive/amharic_distillation_training_clean.csv
/content/drive/MyDrive/predictions_template.csv
/content/drive/MyDrive/eval_prompts_60.csv


In [8]:
import pandas as pd
import os

TRAIN_PATH = "/content/drive/MyDrive/amharic_distillation_training_clean.csv"
EVAL_PATH = "/content/drive/MyDrive/eval_prompts_60.csv"
TEMPLATE_PATH = "/content/drive/MyDrive/predictions_template.csv"

# Load files
train_df = pd.read_csv(TRAIN_PATH)
eval_df = pd.read_csv(EVAL_PATH)
template_df = pd.read_csv(TEMPLATE_PATH)

print("=" * 60)
print("DATASET CHECK")
print("=" * 60)

print(f"Training examples : {len(train_df)}")
print(f"Evaluation prompts: {len(eval_df)}")
print(f"Template rows     : {len(template_df)}")

print("\nTraining columns:")
print(train_df.columns.tolist())

print("\nEvaluation columns:")
print(eval_df.columns.tolist())

print("\nTemplate columns:")
print(template_df.columns.tolist())

print("\nTraining task distribution:")
print(train_df["task"].value_counts())

print("\nFirst training example:")
display(train_df.head(1))

print("\nFirst evaluation example:")
display(eval_df.head(1))

print("\nTemplate:")
display(template_df.head())

DATASET CHECK
Training examples : 1843
Evaluation prompts: 60
Template rows     : 60

Training columns:
['id', 'prompt', 'teacher_response', 'task', 'source']

Evaluation columns:
['eval_id', 'task', 'direction', 'prompt']

Template columns:
['eval_id', 'task', 'base_output', 'distilled_output', 'base_seconds', 'base_new_tokens', 'distilled_seconds', 'distilled_new_tokens', 'teacher_output']

Training task distribution:
task
closed_book              425
summarization            422
reading_comprehension    405
translation_am_en        268
translation_en_am        263
translation               60
Name: count, dtype: int64

First training example:


,id,prompt,teacher_response,task,source
0,distill_0001,የሚከተለውን እንግሊዝኛ ዓረፍተ ነገር ወደ አማርኛ ተርጉም፡\n\nPubli...,የሕዝብ ፖሊሲ ተንታኞች እንደሚያጎሉት በማህበረሰብ የሚመሩ የውሃ አስተዳደ...,translation_en_am,student_train.csv



First evaluation example:


,eval_id,task,direction,prompt
0,E01,closed_book_knowledge,NaN,በ1960ዎቹ አብዛኞቹ የአፍሪካ ሀገራት በአንድ ጊዜ የነፃነት ባለቤት ሊሆ...



Template:


,eval_id,task,base_output,distilled_output,base_seconds,base_new_tokens,distilled_seconds,distilled_new_tokens,teacher_output
0,E01,closed_book_knowledge,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E02,closed_book_knowledge,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E03,closed_book_knowledge,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E04,closed_book_knowledge,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E05,closed_book_knowledge,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
import pandas as pd
import unicodedata
import re
from difflib import SequenceMatcher

TRAIN_PATH = "/content/drive/MyDrive/amharic_distillation_training_clean.csv"
EVAL_PATH = "/content/drive/MyDrive/eval_prompts_60.csv"

train_df = pd.read_csv(TRAIN_PATH)
eval_df = pd.read_csv(EVAL_PATH)


def normalize_prompt(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


train_norm = train_df["prompt"].map(normalize_prompt)
eval_norm = eval_df["prompt"].map(normalize_prompt)

exact_matches = []
near_matches = []

for eval_idx, eval_prompt in enumerate(eval_norm):
    for train_idx, train_prompt in enumerate(train_norm):

        similarity = SequenceMatcher(
            None,
            eval_prompt,
            train_prompt
        ).ratio()

        if similarity == 1.0:
            exact_matches.append(
                (eval_idx, train_idx, similarity)
            )

        elif similarity >= 0.90:
            near_matches.append(
                (eval_idx, train_idx, similarity)
            )


print("=" * 60)
print("EVALUATION LEAKAGE CHECK")
print("=" * 60)

print(f"Training examples checked : {len(train_df)}")
print(f"Evaluation prompts checked: {len(eval_df)}")
print(f"Exact matches             : {len(exact_matches)}")
print(f"Near matches (>= 0.90)    : {len(near_matches)}")


if near_matches:
    print("\nPotential leaked / near-duplicate examples:")

    for eval_idx, train_idx, similarity in sorted(
        near_matches,
        key=lambda x: -x[2]
    )[:20]:

        print("\n" + "-" * 60)
        print(f"Similarity: {similarity:.3f}")
        print(f"Eval ID   : {eval_df.iloc[eval_idx]['eval_id']}")
        print(f"Train ID  : {train_df.iloc[train_idx]['id']}")
        print(f"\nEVAL:\n{eval_df.iloc[eval_idx]['prompt']}")
        print(f"\nTRAIN:\n{train_df.iloc[train_idx]['prompt']}")

EVALUATION LEAKAGE CHECK
Training examples checked : 1843
Evaluation prompts checked: 60
Exact matches             : 0
Near matches (>= 0.90)    : 0


In [10]:
FINAL_TRAIN_PATH = "/content/drive/MyDrive/amharic_distillation_training_final.csv"

train_df.to_csv(FINAL_TRAIN_PATH, index=False)

print("=" * 60)
print("FINAL TRAINING DATASET")
print("=" * 60)
print(f"Saved to: {FINAL_TRAIN_PATH}")
print(f"Examples: {len(train_df)}")
print(f"Evaluation leakage removed: 0")

print("\nTask distribution:")
print(train_df["task"].value_counts())

FINAL TRAINING DATASET
Saved to: /content/drive/MyDrive/amharic_distillation_training_final.csv
Examples: 1843
Evaluation leakage removed: 0

Task distribution:
task
closed_book              425
summarization            422
reading_comprehension    405
translation_am_en        268
translation_en_am        263
translation               60
Name: count, dtype: int64


In [11]:
import torch
import gc
import time
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# Clean GPU memory
gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("LOADING BASE STUDENT")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print(f"Model: {MODEL_ID}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {next(model.parameters()).device}")
print(f"Dtype: {next(model.parameters()).dtype}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

LOADING BASE STUDENT


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-0.5B-Instruct
Parameters: 494,032,768
Device: cuda:0
Dtype: torch.float16
GPU memory allocated: 0.93 GB
GPU memory reserved:  0.95 GB


In [12]:
import time
import torch
import pandas as pd

eval_df = pd.read_csv(EVAL_PATH)

base_outputs = []
base_times = []
base_new_tokens = []

print("=" * 60)
print("BASE STUDENT EVALUATION")
print("=" * 60)
print(f"Prompts: {len(eval_df)}")
print("Decoding: greedy")
print("max_new_tokens: 256")
print()

for i, row in eval_df.iterrows():

    prompt = str(row["prompt"])

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    # Only keep newly generated tokens
    input_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0, input_length:]

    output_text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    base_outputs.append(output_text)
    base_times.append(elapsed)
    base_new_tokens.append(len(new_tokens))

    print(
        f"[{i+1:02d}/60] "
        f"{row['eval_id']} | "
        f"{elapsed:.2f}s | "
        f"{len(new_tokens)} tokens"
    )

print("\n" + "=" * 60)
print("BASE EVALUATION COMPLETE")
print("=" * 60)

print(f"Total time: {sum(base_times):.2f}s")
print(f"Average time/answer: {sum(base_times)/len(base_times):.2f}s")
print(f"Average new tokens: {sum(base_new_tokens)/len(base_new_tokens):.1f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BASE STUDENT EVALUATION
Prompts: 60
Decoding: greedy
max_new_tokens: 256

[01/60] E01 | 10.68s | 256 tokens
[02/60] E02 | 9.77s | 256 tokens
[03/60] E03 | 9.60s | 256 tokens
[04/60] E04 | 8.84s | 256 tokens
[05/60] E05 | 9.52s | 256 tokens
[06/60] E06 | 9.64s | 256 tokens
[07/60] E07 | 9.16s | 256 tokens
[08/60] E08 | 9.55s | 256 tokens
[09/60] E09 | 9.78s | 256 tokens
[10/60] E10 | 9.99s | 256 tokens
[11/60] E11 | 9.11s | 256 tokens
[12/60] E12 | 9.77s | 256 tokens
[13/60] E13 | 9.76s | 256 tokens
[14/60] E14 | 3.47s | 102 tokens
[15/60] E15 | 9.75s | 256 tokens
[16/60] E16 | 8.76s | 256 tokens
[17/60] E17 | 9.80s | 256 tokens
[18/60] E18 | 9.70s | 256 tokens
[19/60] E19 | 8.95s | 256 tokens
[20/60] E20 | 9.62s | 256 tokens
[21/60] E21 | 9.71s | 256 tokens
[22/60] E22 | 9.27s | 256 tokens
[23/60] E23 | 9.32s | 256 tokens
[24/60] E24 | 9.65s | 256 tokens
[25/60] E25 | 9.46s | 256 tokens
[26/60] E26 | 9.01s | 256 tokens
[27/60] E27 | 4.34s | 97 tokens
[28/60] E28 | 8.97s | 256 tokens
[2

In [13]:
# Create a copy of the required submission template
results_df = template_df.copy()

results_df["base_output"] = base_outputs
results_df["base_seconds"] = base_times
results_df["base_new_tokens"] = base_new_tokens

print("=" * 60)
print("BASE RESULTS SAVED IN MEMORY")
print("=" * 60)

print(results_df[[
    "eval_id",
    "task",
    "base_output",
    "base_seconds",
    "base_new_tokens"
]].head(10).to_string(index=False))

BASE RESULTS SAVED IN MEMORY
eval_id                  task                                                                                                                                                                                                                                       base_output  base_seconds  base_new_tokens
    E01 closed_book_knowledge                                                           በመስከት መልክት ሁምንድን እና ክlest ካርጋይ ዝርጋይ የተሰባት ደረጃ የሚያየናይ የሚያደጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያ     10.675411              256
    E02 closed_book_knowledge                                            የስምጥ ሸለቆ ሀይቆች በትናትና በዝናብ ውሃ ደረጃቸውን ጠብቀው የሚቆዩበትን ተፈጥሯዊ ሂደት አብራራ በዚህ በአማርኛ ዝልክል በግጠና ይህ የተገባቁሉ መሰረጡ በአማርኛ ዝልክል በግጠና ይህ የተገባቁሉ መሰረጡ በአማርኛ ዝልክል በግጠና ይህ የተገባቁሉ መሰረጡ በአማርኛ ዝልክል በግጠና ይህ የተገባቁሉ መሰረጡ በአማርኛ ዝ      9.767426              256
    E03 closed_book_knowledge                                 

In [14]:
import numpy as np

# Use the tokenizer already loaded
def format_training_example(row):
    messages = [
        {
            "role": "user",
            "content": str(row["prompt"])
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = prompt_text + str(row["teacher_response"])

    return full_text


lengths = []

for _, row in train_df.iterrows():
    text = format_training_example(row)
    token_count = len(tokenizer(text)["input_ids"])
    lengths.append(token_count)

lengths = np.array(lengths)

print("=" * 60)
print("TRAINING SEQUENCE LENGTH ANALYSIS")
print("=" * 60)

print(f"Examples: {len(lengths)}")
print(f"Minimum:  {lengths.min()} tokens")
print(f"Mean:     {lengths.mean():.1f} tokens")
print(f"Median:   {np.median(lengths):.1f} tokens")
print(f"90th %:   {np.percentile(lengths, 90):.1f} tokens")
print(f"95th %:   {np.percentile(lengths, 95):.1f} tokens")
print(f"99th %:   {np.percentile(lengths, 99):.1f} tokens")
print(f"Maximum:  {lengths.max()} tokens")

for limit in [512, 768, 1024, 1536, 2048]:
    percentage = (lengths <= limit).mean() * 100
    print(f"Within {limit:4d} tokens: {percentage:6.2f}%")

TRAINING SEQUENCE LENGTH ANALYSIS
Examples: 1843
Minimum:  62 tokens
Mean:     344.4 tokens
Median:   351.0 tokens
90th %:   571.0 tokens
95th %:   579.9 tokens
99th %:   594.2 tokens
Maximum:  844 tokens
Within  512 tokens:  80.85%
Within  768 tokens:  99.89%
Within 1024 tokens: 100.00%
Within 1536 tokens: 100.00%
Within 2048 tokens: 100.00%


In [15]:
from datasets import Dataset

MAX_LENGTH = 1024

train_records = []

for _, row in train_df.iterrows():

    prompt = str(row["prompt"])
    teacher_response = str(row["teacher_response"])

    # User portion
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize prompt and teacher response separately
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False
    )["input_ids"]

    response_ids = tokenizer(
        teacher_response,
        add_special_tokens=False
    )["input_ids"]

    # Add EOS to the teacher response
    response_ids = response_ids + [tokenizer.eos_token_id]

    # Keep the complete sequence within MAX_LENGTH
    available_response_tokens = MAX_LENGTH - len(prompt_ids)

    if available_response_tokens <= 0:
        continue

    response_ids = response_ids[:available_response_tokens]

    input_ids = prompt_ids + response_ids

    # -100 means "do not calculate loss on this token"
    labels = (
        [-100] * len(prompt_ids)
        + response_ids
    )

    attention_mask = [1] * len(input_ids)

    train_records.append({
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    })

train_dataset = Dataset.from_list(train_records)

print("=" * 60)
print("RESPONSE-ONLY TRAINING DATASET")
print("=" * 60)

print(f"Original examples : {len(train_df)}")
print(f"Training examples : {len(train_dataset)}")
print(f"Max sequence len  : {MAX_LENGTH}")

# Check how many tokens actually contribute to the loss
response_lengths = [
    sum(1 for x in example["labels"] if x != -100)
    for example in train_records
]

print(f"Average response tokens: {sum(response_lengths)/len(response_lengths):.1f}")
print(f"Maximum response tokens: {max(response_lengths)}")

print("\nFirst example:")
print("Input tokens :", len(train_dataset[0]["input_ids"]))
print("Loss tokens  :", sum(x != -100 for x in train_dataset[0]["labels"]))

RESPONSE-ONLY TRAINING DATASET
Original examples : 1843
Training examples : 1843
Max sequence len  : 1024
Average response tokens: 119.1
Maximum response tokens: 635

First example:
Input tokens : 219
Loss tokens  : 125


In [16]:
import torch
import gc

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

# Clean memory from the baseline model
del model
gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("CONFIGURING QLoRA")
print("=" * 60)

# 4-bit quantization for the base model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

# Prepare the quantized model for LoRA training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("\nGPU memory:")
print(
    f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)
print(
    f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

CONFIGURING QLoRA


/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359

GPU memory:
Allocated: 0.70 GB
Reserved:  1.46 GB


In [17]:
from dataclasses import dataclass
from typing import List, Dict, Any
import torch


@dataclass
class ResponseOnlyDataCollator:

    tokenizer: Any
    label_pad_token_id: int = -100

    def __call__(
        self,
        features: List[Dict[str, Any]]
    ) -> Dict[str, torch.Tensor]:

        max_length = max(
            len(feature["input_ids"])
            for feature in features
        )

        input_ids = []
        attention_masks = []
        labels = []

        for feature in features:

            padding_length = (
                max_length - len(feature["input_ids"])
            )

            input_ids.append(
                feature["input_ids"]
                + [self.tokenizer.pad_token_id] * padding_length
            )

            attention_masks.append(
                feature["attention_mask"]
                + [0] * padding_length
            )

            labels.append(
                feature["labels"]
                + [self.label_pad_token_id] * padding_length
            )

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                attention_masks,
                dtype=torch.long
            ),
            "labels": torch.tensor(
                labels,
                dtype=torch.long
            )
        }


# Qwen normally has an EOS token but may not have a pad token.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

data_collator = ResponseOnlyDataCollator(
    tokenizer=tokenizer
)

# Test one small batch
test_batch = data_collator([
    train_dataset[0],
    train_dataset[1],
    train_dataset[2],
])

print("=" * 60)
print("DATA COLLATOR CHECK")
print("=" * 60)

print("input_ids shape     :", test_batch["input_ids"].shape)
print("attention shape     :", test_batch["attention_mask"].shape)
print("labels shape        :", test_batch["labels"].shape)

print(
    "Masked label tokens :",
    (test_batch["labels"] == -100).sum().item()
)

print(
    "Loss label tokens   :",
    (test_batch["labels"] != -100).sum().item()
)

DATA COLLATOR CHECK
input_ids shape     : torch.Size([3, 426])
attention shape     : torch.Size([3, 426])
labels shape        : torch.Size([3, 426])
Masked label tokens : 844
Loss label tokens   : 434


In [18]:
from transformers import TrainingArguments

OUTPUT_DIR = "/content/drive/MyDrive/qwen25_05b_amharic_qlora"

# Disable KV cache during training
model.config.use_cache = False

# Needed with gradient checkpointing + PEFT
model.enable_input_require_grads()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    # LoRA optimization
    learning_rate=2e-4,
    weight_decay=0.0,
    warmup_ratio=0.05,
    max_grad_norm=0.3,

    # Memory / speed
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",

    # Logging / checkpoints
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=2,

    # Reproducibility
    seed=42,
    data_seed=42,

    # Dataset already contains tensors/lists handled by our collator
    remove_unused_columns=False,

    # No external logging
    report_to="none"
)

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)

print(f"Epochs:                    {training_args.num_train_epochs}")
print(f"Batch size per GPU:       {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation:    {training_args.gradient_accumulation_steps}")
print(f"Effective batch size:     {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate:             {training_args.learning_rate}")
print(f"Warmup ratio:              {training_args.warmup_ratio}")
print(f"Max sequence length:       {MAX_LENGTH}")
print(f"Gradient checkpointing:    {training_args.gradient_checkpointing}")
print(f"Optimizer:                 {training_args.optim}")
print(f"FP16:                      {training_args.fp16}")
print(f"Output:                    {OUTPUT_DIR}")

TRAINING CONFIGURATION
Epochs:                    3
Batch size per GPU:       4
Gradient accumulation:    4
Effective batch size:     16
Learning rate:             0.0002
Warmup ratio:              0.05
Max sequence length:       1024
Gradient checkpointing:    True
Optimizer:                 OptimizerNames.PAGED_ADAMW_8BIT
FP16:                      True
Output:                    /content/drive/MyDrive/qwen25_05b_amharic_qlora


In [19]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("=" * 60)
print("TRAINER READY")
print("=" * 60)

print(f"Training examples : {len(train_dataset)}")
print(f"Trainable params  : 2,162,688")
print(f"Effective batch   : 16")
print(f"Epochs            : 3")
print(f"Total examples    : {len(train_dataset) * 3}")

TRAINER READY
Training examples : 1843
Trainable params  : 2,162,688
Effective batch   : 16
Epochs            : 3
Total examples    : 5529


In [20]:
import time
import torch

print("=" * 60)
print("STARTING QLoRA TRAINING")
print("=" * 60)

torch.cuda.reset_peak_memory_stats()

train_start = time.perf_counter()

train_result = trainer.train()

train_seconds = time.perf_counter() - train_start

peak_memory_gb = (
    torch.cuda.max_memory_allocated() / 1024**3
)

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(f"Training time:        {train_seconds / 60:.2f} minutes")
print(f"Final training loss:   {train_result.training_loss:.4f}")
print(f"Peak GPU memory:       {peak_memory_gb:.2f} GB")
print(f"Training samples:      {len(train_dataset)}")
print(f"Epochs:                3")

STARTING QLoRA TRAINING


Step,Training Loss
25,2.370200
50,2.081600
75,1.861500
100,1.640500
125,1.612200
150,1.621100


Step,Training Loss
25,2.370200
50,2.081600
75,1.861500
100,1.640500
125,1.612200
150,1.621100
175,1.554800
200,1.661700
225,1.627800
250,1.576600



TRAINING COMPLETE
Training time:        17.73 minutes
Final training loss:   1.6830
Peak GPU memory:       7.00 GB
Training samples:      1843
Epochs:                3


In [21]:
import os
import torch

FINAL_ADAPTER_DIR = "/content/drive/MyDrive/qwen25_05b_amharic_qlora/final_adapter"

print("=" * 60)
print("SAVING FINAL QLoRA ADAPTER")
print("=" * 60)

trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print(f"Saved to: {FINAL_ADAPTER_DIR}")

print("\nSaved files:")
for filename in sorted(os.listdir(FINAL_ADAPTER_DIR)):
    filepath = os.path.join(FINAL_ADAPTER_DIR, filename)
    size_mb = os.path.getsize(filepath) / (1024 ** 2)
    print(f"  {filename:<40} {size_mb:8.2f} MB")

SAVING FINAL QLoRA ADAPTER
Saved to: /content/drive/MyDrive/qwen25_05b_amharic_qlora/final_adapter

Saved files:
  README.md                                    0.00 MB
  adapter_config.json                          0.00 MB
  adapter_model.safetensors                    8.27 MB
  added_tokens.json                            0.00 MB
  chat_template.jinja                          0.00 MB
  merges.txt                                   1.59 MB
  special_tokens_map.json                      0.00 MB
  tokenizer.json                              10.89 MB
  tokenizer_config.json                        0.00 MB
  training_args.bin                            0.01 MB
  vocab.json                                   2.65 MB


In [22]:
import torch
import time

print("=" * 60)
print("PREPARING DISTILLED MODEL")
print("=" * 60)

model.eval()

# Re-enable KV cache for faster generation
model.config.use_cache = True

# Make sure gradients are disabled
for param in model.parameters():
    param.requires_grad = False

print("Model set to evaluation mode.")
print(f"Device: {next(model.parameters()).device}")
print(f"Dtype: {next(model.parameters()).dtype}")

print("\nGPU memory:")
print(
    f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)
print(
    f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

PREPARING DISTILLED MODEL
Model set to evaluation mode.
Device: cuda:0
Dtype: torch.float32

GPU memory:
Allocated: 0.71 GB
Reserved:  9.34 GB


In [23]:
import time
import torch
import pandas as pd

eval_df = pd.read_csv(EVAL_PATH)

distilled_outputs = []
distilled_times = []
distilled_new_tokens = []

print("=" * 60)
print("DISTILLED STUDENT EVALUATION")
print("=" * 60)
print(f"Prompts: {len(eval_df)}")
print("Decoding: greedy")
print("max_new_tokens: 256")
print("Batch size: 1")
print()

torch.cuda.reset_peak_memory_stats()

eval_start = time.perf_counter()

for i, row in eval_df.iterrows():

    prompt = str(row["prompt"])

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    input_length = inputs["input_ids"].shape[1]

    new_tokens = output_ids[0, input_length:]

    output_text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    distilled_outputs.append(output_text)
    distilled_times.append(elapsed)
    distilled_new_tokens.append(len(new_tokens))

    print(
        f"[{i+1:02d}/60] "
        f"{row['eval_id']} | "
        f"{elapsed:.2f}s | "
        f"{len(new_tokens)} tokens"
    )

torch.cuda.synchronize()

total_eval_seconds = time.perf_counter() - eval_start

peak_eval_memory_gb = (
    torch.cuda.max_memory_allocated() / 1024**3
)

print("\n" + "=" * 60)
print("DISTILLED EVALUATION COMPLETE")
print("=" * 60)

print(
    f"Total time:          {total_eval_seconds:.2f}s"
)

print(
    f"Average time/answer: "
    f"{sum(distilled_times) / len(distilled_times):.2f}s"
)

print(
    f"Average new tokens:  "
    f"{sum(distilled_new_tokens) / len(distilled_new_tokens):.1f}"
)

print(
    f"Peak GPU memory:     "
    f"{peak_eval_memory_gb:.2f} GB"
)

DISTILLED STUDENT EVALUATION
Prompts: 60
Decoding: greedy
max_new_tokens: 256
Batch size: 1



/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[01/60] E01 | 33.95s | 196 tokens
[02/60] E02 | 25.23s | 256 tokens
[03/60] E03 | 25.62s | 256 tokens
[04/60] E04 | 2.61s | 28 tokens
[05/60] E05 | 25.35s | 256 tokens
[06/60] E06 | 25.08s | 256 tokens
[07/60] E07 | 4.28s | 47 tokens
[08/60] E08 | 25.02s | 256 tokens
[09/60] E09 | 24.83s | 256 tokens
[10/60] E10 | 1.84s | 20 tokens
[11/60] E11 | 25.06s | 256 tokens
[12/60] E12 | 25.16s | 256 tokens
[13/60] E13 | 25.50s | 256 tokens
[14/60] E14 | 24.37s | 256 tokens
[15/60] E15 | 2.53s | 20 tokens
[16/60] E16 | 6.58s | 72 tokens
[17/60] E17 | 15.23s | 155 tokens
[18/60] E18 | 10.94s | 110 tokens
[19/60] E19 | 12.51s | 125 tokens
[20/60] E20 | 2.68s | 29 tokens
[21/60] E21 | 7.43s | 70 tokens
[22/60] E22 | 4.01s | 42 tokens
[23/60] E23 | 11.47s | 116 tokens
[24/60] E24 | 6.67s | 73 tokens
[25/60] E25 | 4.32s | 37 tokens
[26/60] E26 | 4.26s | 45 tokens
[27/60] E27 | 13.26s | 131 tokens
[28/60] E28 | 6.81s | 70 tokens
[29/60] E29 | 1.95s | 16 tokens
[30/60] E30 | 20.01s | 209 tokens
[31/60

In [24]:
import pandas as pd
import numpy as np

# Start from the original required template
final_results = template_df.copy()

# Base results
final_results["base_output"] = base_outputs
final_results["base_seconds"] = base_times
final_results["base_new_tokens"] = base_new_tokens

# Distilled results
final_results["distilled_output"] = distilled_outputs
final_results["distilled_seconds"] = distilled_times
final_results["distilled_new_tokens"] = distilled_new_tokens

# Teacher output is optional.
# Leave it empty unless you already have teacher outputs
# for these exact 60 evaluation prompts.
final_results["teacher_output"] = ""

FINAL_RESULTS_PATH = (
    "/content/drive/MyDrive/predictions_final.csv"
)

final_results.to_csv(
    FINAL_RESULTS_PATH,
    index=False
)

print("=" * 60)
print("FINAL PREDICTIONS FILE")
print("=" * 60)

print(f"Saved to: {FINAL_RESULTS_PATH}")
print(f"Rows:     {len(final_results)}")
print(f"Columns:  {len(final_results.columns)}")

print("\nColumns:")
print(final_results.columns.tolist())

print("\nMissing values:")
print(final_results.isna().sum())

print("\nFirst 3 results:")
display(
    final_results[
        [
            "eval_id",
            "task",
            "base_output",
            "distilled_output",
            "base_seconds",
            "distilled_seconds"
        ]
    ].head(3)
)

FINAL PREDICTIONS FILE
Saved to: /content/drive/MyDrive/predictions_final.csv
Rows:     60
Columns:  9

Columns:
['eval_id', 'task', 'base_output', 'distilled_output', 'base_seconds', 'base_new_tokens', 'distilled_seconds', 'distilled_new_tokens', 'teacher_output']

Missing values:
eval_id                 0
task                    0
base_output             0
distilled_output        0
base_seconds            0
base_new_tokens         0
distilled_seconds       0
distilled_new_tokens    0
teacher_output          0
dtype: int64

First 3 results:


,eval_id,task,base_output,distilled_output,base_seconds,distilled_seconds
0,E01,closed_book_knowledge,በመስከት መልክት ሁምንድን እና ክlest ካርጋይ ዝርጋይ የተሰባት ደረጃ ...,የ1960ች እንደሚያጠቀም ስርስተኛ መሰረት (የመጣል ማግባዊ ምክራዊ ፕሮጀ...,10.675411,33.945866
1,E02,closed_book_knowledge,የስምጥ ሸለቆ ሀይቆች በትናትና በዝናብ ውሃ ደረጃቸውን ጠብቀው የሚቆዩበት...,የስምጥ ሸለቆ ሀይቆች በትናትና በዝናብ ውሃ ደረጃቸውን የተሰጠበት የማካል...,9.767426,25.233980
2,E03,closed_book_knowledge,የፋሲለደስ ግንብ በየትኛዋ የኢትዮጵያ ታሪanca ሁይና እanzi ካርቃላች...,የፋሲለደስ ስራት በመጠበቅ የኢትዮጵያ ታሪካዊ ከተማ ውስጥ መረጃ ያላቸው ...,9.601117,25.619943


In [25]:
import pandas as pd

df = pd.read_csv(FINAL_RESULTS_PATH)

# Add useful derived metrics
df["base_tokens_per_second"] = (
    df["base_new_tokens"] / df["base_seconds"]
)

df["distilled_tokens_per_second"] = (
    df["distilled_new_tokens"] / df["distilled_seconds"]
)

print("=" * 60)
print("EVALUATION SUMMARY BY TASK")
print("=" * 60)

summary = (
    df.groupby("task")
    .agg(
        prompts=("eval_id", "count"),

        base_avg_seconds=("base_seconds", "mean"),
        distilled_avg_seconds=("distilled_seconds", "mean"),

        base_avg_tokens=("base_new_tokens", "mean"),
        distilled_avg_tokens=("distilled_new_tokens", "mean"),

        base_avg_tok_s=("base_tokens_per_second", "mean"),
        distilled_avg_tok_s=("distilled_tokens_per_second", "mean"),
    )
    .round(2)
)

display(summary)

print("\n" + "=" * 60)
print("OVERALL")
print("=" * 60)

print(f"Prompts: {len(df)}")
print(
    f"Base average time:      "
    f"{df['base_seconds'].mean():.2f}s"
)
print(
    f"Distilled average time: "
    f"{df['distilled_seconds'].mean():.2f}s"
)
print(
    f"Base average tokens:      "
    f"{df['base_new_tokens'].mean():.1f}"
)
print(
    f"Distilled average tokens: "
    f"{df['distilled_new_tokens'].mean():.1f}"
)
print(
    f"Base average tok/s:       "
    f"{df['base_tokens_per_second'].mean():.2f}"
)
print(
    f"Distilled average tok/s:  "
    f"{df['distilled_tokens_per_second'].mean():.2f}"
)

EVALUATION SUMMARY BY TASK


,prompts,base_avg_seconds,distilled_avg_seconds,base_avg_tokens,distilled_avg_tokens,base_avg_tok_s,distilled_avg_tok_s
task,,,,,,,
closed_book_knowledge,15,9.23,19.76,245.73,191.40,26.80,9.88
instruction_following,15,9.06,8.54,245.40,86.67,26.95,10.06
summarization,15,6.33,22.64,167.07,228.73,26.51,10.09
translation,15,3.39,5.82,92.73,59.00,28.06,10.11



OVERALL
Prompts: 60
Base average time:      7.00s
Distilled average time: 14.19s
Base average tokens:      187.7
Distilled average tokens: 141.4
Base average tok/s:       27.08
Distilled average tok/s:  10.03


In [27]:
import pandas as pd

results = pd.read_csv(FINAL_RESULTS_PATH)
eval_prompts = pd.read_csv(EVAL_PATH)

df = results.merge(
    eval_prompts[["eval_id", "prompt", "direction"]],
    on="eval_id",
    how="left"
)

print("=" * 80)
print("QUALITATIVE BASE vs DISTILLED COMPARISON")
print("=" * 80)

for task in ["closed_book_knowledge",
             "instruction_following",
             "translation",
             "summarization"]:

    print("\n" + "=" * 80)
    print(f"TASK: {task}")
    print("=" * 80)

    task_df = df[df["task"] == task].head(3)

    for _, row in task_df.iterrows():

        print("\n" + "-" * 80)
        print(f"Evaluation ID: {row['eval_id']}")

        if pd.notna(row["direction"]):
            print(f"Direction:     {row['direction']}")

        print("-" * 80)

        print("\nPROMPT:")
        print(row["prompt"])

        print("\nBASE OUTPUT:")
        print(row["base_output"])

        print("\nDISTILLED OUTPUT:")
        print(row["distilled_output"])

QUALITATIVE BASE vs DISTILLED COMPARISON

TASK: closed_book_knowledge

--------------------------------------------------------------------------------
Evaluation ID: E01
--------------------------------------------------------------------------------

PROMPT:
በ1960ዎቹ አብዛኞቹ የአፍሪካ ሀገራት በአንድ ጊዜ የነፃነት ባለቤት ሊሆኑ የቻሉት ለምንድን ነው?

BASE OUTPUT:
በመስከት መልክት ሁምንድን እና ክlest ካርጋይ ዝርጋይ የተሰባት ደረጃ የሚያየናይ የሚያደጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያጋጋ የሚያ

DISTILLED OUTPUT:
የ1960ች እንደሚያጠቀም ስርስተኛ መሰረት (የመጣል ማግባዊ ምክራዊ ፕሮጀክቶች) ይፈጥር ያላቸው። በአፍሪካ ሀገራት ተማይና በአንድ ጊዜ የነፃነት ባለቤት ሊሆኑ የቻሉት ለምንድን ነው። በአንድ ጊዜ የነፃነት ባለቤት ሊሆኑ የቻሉት ለምንድን ነው።

--------------------------------------------------------------------------------
Evaluation ID: E02
--------------------------------------------------------------------------------

PROMPT:
የስምጥ ሸለቆ ሀይቆች በትናትና በዝናብ ውሃ ደረጃቸውን ጠብቀው የሚቆዩበትን ተፈጥሯዊ ሂደት አብራራ።

BASE OUTPUT:
የስምጥ ሸለቆ ሀይቆች በትናትና በዝናብ ውሃ ደረጃቸውን ጠ